# Subtitles Showcase

This notebook demonstrates how to extract and process text from `.srt` subtitle files, clean the text, remove duplicates, and finally extract vocabulary tokens (verbs and other words) using the Natural Language API.

In [1]:
%load_ext autoreload

In [2]:
%autoreload 2
from setup_imports import *  # noqa: F401,F403


In [3]:


from src.subtitles import read_subtitles, process_subtitles, get_subtitle_tokens
from src.nlp import get_text_tokens, get_verbs_and_vocab
from src.phrases.phrase_model import get_unique_tokens_from_phrases, Phrase
from src.phrases.generation import generate_phrases_from_vocab_dict
from src.phrases.search import get_phrases_by_collection, find_phrases_by_vocab_dict
from src.utils import save_text_file, load_text_file, save_json, load_json

## 1. Reading Subtitles
We use `pysrt` to read the subtitles from the `data/subtitles/` directory.

In [12]:
subtitle_file = "../data/subtitles/Trouble_sv_only.srt"

try:
    subs = read_subtitles(subtitle_file)
    print(f"Successfully loaded {len(subs)} subtitle entries.")
    print("\nFirst 3 entries:")
    for i in range(min(3, len(subs))):
        print(f"{i+1}: {subs[i].text}")
except Exception as e:
    print(f"Error reading subtitles: {e}")

Successfully loaded 1543 subtitle entries.

First 3 entries:
1: [visslar en melodi]
2: Men snälla, tyst.
3: Du väntar här. När jag messar dig
kommer du in med resten av grejerna.


## 2. Processing Subtitles
We will now clean the subtitles by removing Closed Captioning (CC) info enclosed in square brackets `[like this]`, replace newlines with spaces, strip extra whitespace, and remove duplicates to get a unique list of phrases.

In [13]:
unique_phrases = process_subtitles(subs, language_code="sv", split_on_space=True)

print(f"Total unique phrases extracted: {len(unique_phrases)}")
print("\nSample of 10 unique phrases:")
for phrase in unique_phrases[:100]:
    print(f"{phrase}")

Total unique phrases extracted: 1313

Sample of 10 unique phrases:
Men snälla tyst
Du väntar här När jag messar dig kommer du in med resten av grejerna
Ja Capite
Ska vi sticka
Vad är det här för nåt
Det är ett kilo det här Vi sa 30  Mm
Vi har dem Men pengarna först
Vi står utanför Bamboo Bamboo
Det är bekräftat de misstänkta befinner sig i lokalen
Kör
Enhet A vänster Enhet B bakvägen
Ja Uppfattat
Portkoden
1490
Vänster Vänster Vänster
Ja grabbar det här kan ni In och plocka dem
så tar vi en AW på Stopet sen
Då kallar jag in grejerna
Vapen Vapen
Vad händer Vad fan händer
Nej
Angripare avväpnad Vi avancerar
Polis Polis Visa händerna
Ner på golvet Släpp vapnen Den tog i västen Det är lugnt
Nu tar vi dem
Ner på golvet Ner på golvet
Händerna över huvudet Håll käften
Var fan är alla droger
Det saknas en massa Var fan är resten
Hej
Hej hej ta det lugnt Okej okej
Chilla okej okej Vänta
Här
Här Ta grejerna Skjut inte
Åh shit
Mina damer och herrar detta är kapten Conny Rundkvist från cockpit
Som

## 3. Extracting Tokens
Using `src.nlp`, we'll run a sample of these phrases through the Google Cloud Natural Language API to extract lemmas and group them into `verbs` and `vocab`.

In [14]:
# To avoid long API wait times in this showcase, we'll process just the first 20 phrases.
sample_phrases = unique_phrases[:20]

print(f"Extracting vocabulary from {len(unique_phrases)} phrases...")
vocab_dict = get_verbs_and_vocab(unique_phrases, language="sv-se")

print("\n--- Extracted Verbs ---")
print(vocab_dict.get('verbs', []))

print("\n--- Extracted Vocab (Non-Verbs) ---")
print(vocab_dict.get('vocab', []))

Extracting vocabulary from 1313 phrases...
2026-05-03 14:33:35 - audio-language-trainer - INFO - nlp.py:83 - _load_spacy_model: loaded spaCy model 'sv_core_news_lg' for language 'sv'

--- Extracted Verbs ---
['beställ', 'sova', 'agera', 'flytta', 'släppa', 'akta', 'hjälpa', 'bromsa', 'planera', 'mörda', 'avbryta', 'förstå', 'luka', 'gripina', 'ladda', 'avgöra', 'slå', 'spara', 'ligga', 'går', 'rikta', 'gå', 'avgå', 'sakna', 'klassa', 'ring', 'störa', 'möta', 'flög', 'glöms', 'ge', 'glöm', 'ser', 'inträffa', 'vara', 'lämna', 'raka', 'installera', 'anta', 'förändra', 'köra', 'leda', 'fastnaa', 'gick', 'dra', 'förhöra', 'titta', 'ställa', 'skriva', 'precis', 'spela', 'döda', 'pilla', 'fixaa', 'skola', 'druckit', 'bli', 'försöka', 'igenkända', 'nämna', 'bo', 'sluta', 'grabeb', 'använma', 'gilla', 'hugga', 'transportera', 'förstår', 'aktivera', 'finnas', 'taa', 'kunna', 'jävlar', 'uppmana', 'säga', 'klara', 'skjut', 'upprepa', 'funka', 'erkänn', 'starta', 'snacka', 'hör', 'fan', 'diana', 'h

In [ ]:
p2, m3 = find_phrases_by_vocab_dict({"verbs" : ["ska"], "vocab" : ["problem"]}, language="sv-SE")

In [26]:
p2[0].translations["sv-SE"].verbs

['skola', 'pulla']

In [15]:
p, m = find_phrases_by_vocab_dict(vocab_dict, language="sv-SE")

(y) Authenticated with Google Cloud project: swedish-course


In [16]:
from src.subtitles import get_subtitle_tokens
missing_verbs = m['verbs']
missing_valid_verbs = get_subtitle_tokens(missing_verbs, "sv", min_length=3, pos_list=["verb"])
missing_vocab = m['vocab']
missing_valid_vocab = get_subtitle_tokens(missing_vocab, "sv", min_length=4, pos_list=["noun", "adj", "adv"])

In [17]:
subtitle_vocab_dict = {"verbs" : missing_valid_verbs, "vocab" : missing_valid_vocab}

In [18]:
from utils import save_json, load_json
save_json(subtitle_vocab_dict, file_path="../data/trouble_2024_vocab_dict.json")
#subtitle_vocab_dict = load_json("../data/trouble_2024_vocab_dict.json")

In [39]:
len(subtitle_vocab_dict["verbs"]), len(subtitle_vocab_dict["vocab"])

(119, 264)

# Generate new foreign phrases

In [24]:
# tags=["film::Trouble_2024"]
subset_dict = {"verbs": subtitle_vocab_dict["verbs"][:10], "vocab": subtitle_vocab_dict["vocab"][:50]}


In [ ]:
subtitle_phrases = generate_phrases_from_vocab_dict(subset_dict, language="sv-SE")

In [ ]:
save_text_file(subtitle_phrases, "../data/Trouble_24_subtitle_phrases_10_verbs_50_vocab.txt")

In [ ]:
subtitle_phrases

In [ ]:
ALL_PHRASES = []
for foreign in subtitle_phrases:
    p = Phrase.create_from_foreign(foreign, language="sv-SE", split_on_space=True, tags=["film::Trouble_2024"])
    p.upload()
    ALL_PHRASES.append(p)

In [ ]:
lm1000 = get_phrases_by_collection("LM1000")

In [ ]:
lm1000_str = [(x._get_translation('sv-SE').text) for x in lm1000[:10]]

In [13]:
results = find_phrases_by_vocab_dict(
vocab_dict={"verbs": ["springa", "äta"], "vocab": ["hund", "bil"]},
language="sv-SE",
collection="LM1000")
for p in results:
    print(p.english, p.translations["sv-SE"].text)

2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_syntax: language not supported by Google NLP: sv-SE
2026-04-19 15:12:14 - audio-language-trainer - INFO - nlp.py:189 - extract_lemmas_and_pos: falling back to spaCy for language 'sv-SE'
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:197 - extract_lemmas_and_pos: neither Google NLP nor spaCy could process phrase for language 'sv-SE': Ett bättre alternativ
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_syntax: language not supported by Google NLP: sv-SE
2026-04-19 15:12:14 - audio-language-trainer - INFO - nlp.py:189 - extract_lemmas_and_pos: falling back to spaCy for language 'sv-SE'
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:197 - extract_lemmas_and_pos: neither Google NLP nor spaCy could process phrase for language 'sv-SE': En cykel i låg växel
2026-04-19 15:12:14 - audio-language-trainer - WARNING - nlp.py:151 - analyze_text_synt

{'verbs': [],
 'vocab': [],
 'tokens': ['och',
  'rent',
  'flicka',
  'person',
  'under',
  'för',
  'kort',
  'billig',
  'bättre',
  'färg',
  'pappersark',
  'ljus',
  'förändring',
  'Ett',
  'i',
  'cykel',
  'bänken',
  'genomtänkt',
  'flaska',
  'pojke',
  'varje',
  'glasring',
  'En',
  'växel',
  'svar',
  'alternativ',
  'låg',
  'en',
  'väster']}

In [ ]:
get_

In [ ]:
lm1000_tokens = get_unique_tokens_from_phrases(lm1000, "sv-SE")
lm1000_tokens = list(map(lambda x: x.lower(), lm1000_tokens))

In [ ]:
len(film_tokens)

In [ ]:
len(lm1000_tokens)

In [ ]:
(set(map(lambda x: x[:-1],film_tokens)).difference(set(lm1000_tokens)))

In [ ]:
film_tokens